# Prompt Examples

- https://platform.openai.com/docs/guides/function-calling 참조

- API 호출에서 함수를 설명하고 모델이 하나 이상의 함수를 호출하기 위한 인수를 포함하는 JSON 객체를 출력하도록 지능적으로 선택할 수 있습니다. ***Chat Completions API는 함수를 호출하지 않습니다.** 대신 모델은 코드에서 **함수를 호출하는 데 사용할 수 있는 JSON을 생성**합니다. 

최신 모델( gpt-4o, gpt-4-turbo, 및 gpt-4o-mini)은 함수를 호출해야 하는 시점(입력에 따라 다름)을 감지하고 이전 모델보다 함수 시그니처를 더 면밀히 준수하는 JSON으로 응답하도록 훈련되었습니다. 이 기능에는 잠재적인 위험도 따릅니다. 사용자를 대신하여 세상에 영향을 미치는 작업(이메일 보내기, 온라인에 게시하기, 구매하기 등)을 수행하기 전에 사용자 확인 흐름을 빌드하는 것이 좋습니다.

### 일반적인 사용 사례
함수 호출을 통해 모델에서 구조화된 데이터를 보다 안정적으로 가져올 수 있습니다. 예를 들어, 다음을 수행할 수 있습니다.

- 외부 API를 호출하여 질문에 답변하는 도우미를 만듭니다. 예를 들어, 다음과 같은 함수를 정의합니다.  
send_email(to: string, body: string) 또는 get_current_weather(location: string, unit: 'celsius' | 'fahrenheit')
 
- 자연어를 API 호출로 변환  
예를 들어 "내 최고 고객은 누구입니까?"를 get_customers(min_revenue: int, created_before: string, limit: int)로 convert 하고 내부 API를 호출합니다.

- 텍스트에서 구조화된 데이터 추출  
extract_data(name: string, birthday: string) 라는 함수를 정의하거나 sql_query(query: string) 라는 함수 정의

함수 호출의 기본적인 단계 순서는 다음과 같습니다.

1. 사용자 쿼리와 functions 매개변수 에 정의된 함수 집합으로 모델을 호출합니다.  
2. 모델은 하나 이상의 함수를 호출하도록 선택할 수 있습니다. 그럴 경우, 콘텐츠는 사용자 지정 스키마를 준수하는 문자열화된 JSON 객체가 됩니다(참고: 모델은 매개변수를 혼동할 수 있음).  
3. 코드에서 문자열을 JSON으로 구문 분석하고, 제공된 인수가 있으면 해당 인수로 함수를 호출합니다.  
4. 함수 응답을 새 메시지로 추가하여 모델을 다시 호출하고, 모델이 결과를 사용자에게 요약하도록 합니다. 

### 함수 호출 동작
기본적으로 `tool_choice`의 기본 값은 `tool_choice: "auto"`입니다. 이는 모델이 함수 호출 여부와 호출할 함수를 결정하도록 합니다.

사용 사례에 따라 기본 동작을 맞춤화하는 세 가지 방법을 제공합니다:

1. 모델이 항상 하나 이상의 함수를 호출하도록 강제하려면, `tool_choice: "required"`로 설정할 수 있습니다. 그러면 모델은 호출할 함수(들)를 선택합니다.
2. 모델이 특정 함수만 호출하도록 강제하려면, `tool_choice: {"type": "function", "function": {"name": "my_function"}}`로 설정할 수 있습니다.
3. 함수 호출을 비활성화하고 모델이 사용자에게 표시할 메시지만 생성하도록 강제하려면, `tool_choice: "none"`으로 설정할 수 있습니다.